In [1]:
import torch
import torch.nn as nn

In [2]:
# simple way is to make a wrapper on the top of mask attention
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your    (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts  (x^3)
   [0.22, 0.58, 0.33], # with    
   [0.77, 0.25, 0.10], # one     
   [0.05, 0.80, 0.55]] # step    
)
d_in = inputs.shape[1]
d_out = 2

In [3]:
class CausalSelfAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        attention_scores = queries @ keys.transpose(1,2)
        attention_scores = attention_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], float('-inf'))
        attention_weights = torch.softmax(attention_scores / d_in**0.5, dim=-1)
        attention_weights = self.dropout(attention_weights)

        context_vectors = attention_weights @ values
        return context_vectors


In [4]:

batch = torch.stack((inputs, inputs), dim=0)

In [5]:
# wrapper class
class MultiHeadAttentionWrapper(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList([CausalSelfAttention(d_in, d_out, context_length, dropout, qkv_bias) for _ in range(num_heads)])

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

In [6]:
torch.manual_seed(123)
context_length = batch.shape[1]
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, dropout=0.0, num_heads=2)
context_vectors = mha(batch)
context_vectors

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5866,  0.0071,  0.5869,  0.3214],
         [-0.6293, -0.0621,  0.6184,  0.3825],
         [-0.5670, -0.0838,  0.5474,  0.3575],
         [-0.5519, -0.0979,  0.5319,  0.3423],
         [-0.5295, -0.1077,  0.5074,  0.3481]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5866,  0.0071,  0.5869,  0.3214],
         [-0.6293, -0.0621,  0.6184,  0.3825],
         [-0.5670, -0.0838,  0.5474,  0.3575],
         [-0.5519, -0.0979,  0.5319,  0.3423],
         [-0.5295, -0.1077,  0.5074,  0.3481]]], grad_fn=<CatBackward0>)

In [15]:
# lets implement multi head attention without wrapper class
class MultiHeadAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads) == 0, "d_out must be divisible by num_heads"
        self.d_out = d_out
        self.context_length = context_length
        self.dropout = nn.Dropout(dropout)
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))


    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

        keys = keys.transpose(1,2)
        queries = queries.transpose(1,2)
        values = values.transpose(1,2)

        attention_scores = queries @ keys.transpose(2,3)

        attention_scores = attention_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], float('-inf'))
        attention_weights = torch.softmax(attention_scores / self.head_dim**0.5, dim=-1)
        attention_weights = self.dropout(attention_weights)
        context_vectors = (attention_weights @ values).transpose(1,2)
        context_vectors = context_vectors.contiguous().view(b, num_tokens, self.d_out)
        context_vectors = self.out_proj(context_vectors)
        return context_vectors



In [16]:
torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, dropout=0.0, num_heads=2)
context_vectors = mha(batch)
context_vectors

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)

In [20]:
batch_size, context_length, d_in = batch.shape
d_out = d_in
context_length = 1024
mha = MultiHeadAttention(d_in, d_out, context_length, dropout=0.0, num_heads=3)

In [21]:
mha(batch)

tensor([[[-0.0502,  0.4002, -0.3991],
         [-0.1109,  0.2592, -0.4498],
         [-0.1287,  0.2189, -0.4646],
         [-0.1477,  0.1948, -0.4860],
         [-0.1272,  0.2414, -0.4629],
         [-0.1494,  0.2004, -0.4873]],

        [[-0.0502,  0.4002, -0.3991],
         [-0.1109,  0.2592, -0.4498],
         [-0.1287,  0.2189, -0.4646],
         [-0.1477,  0.1948, -0.4860],
         [-0.1272,  0.2414, -0.4629],
         [-0.1494,  0.2004, -0.4873]]], grad_fn=<ViewBackward0>)